# VLM-DENTAL — Evaluation inventory and missing cohorts
Author: Reza Nadimi <rezaxr14@gmail.com>

After the original zero-shot/SFT/GRPO/rules-skills/speculative workflows, inspect the existing local and Hub evidence and run only missing work. All notebooks share the same result identities. Training and frontier API jobs are never started here. Results go to the existing `Reza-Nadimi/vlm-dental-evaluations` repository under `evaluations/dentex/`; this notebook does not build the paper.

## 1. Evaluation checkout

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "main"  # or the exact reviewed commit containing this patch
if not Path("dental_agent/harness/notebook_eval.py").is_file():
    parent = Path("/kaggle/working" if Path("/kaggle").exists() else "/content" if Path("/content").exists() else Path.cwd())
    work = parent / "VLM-DENTAL-EVALUATION"
    if not work.exists():
        if len(REPO_REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REPO_REF):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)], check=True)
            subprocess.run(["git", "-C", str(work), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
            subprocess.run(["git", "-C", str(work), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(work)], check=True)
    os.chdir(work)
assert Path("dental_agent/harness/notebook_eval.py").is_file(), "Use the checkout containing this patch"
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
print("Checkout:", Path.cwd())

## 2. Dependencies and existing credentials

In [ ]:
INSTALL_DEPENDENCIES = True  # install only in this isolated evaluation runtime
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[api,dev]"], check=True)
from dotenv import load_dotenv
load_dotenv()
for key in ("HF_TOKEN", "HF_ARTIFACT_REPO", "DENTEX_IMAGES_REPO"):
    if os.environ.get(key): continue
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(key)
        elif IS_COLAB:
            from google.colab import userdata
            value = userdata.get(key)
        else: value = None
        if value: os.environ[key] = value
    except Exception: pass
print("Artifacts:", os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models"))

## 3. Profiles and cohorts
Set `EVAL_TARGET_PROFILES` to one profile, a list, or `"all"`. This selects checkpoint cohorts, not a claim that the current hardware supports every numerical runtime. Native BF16 and serving requirements are checked before weights load. Unsupported profiles remain listed as pending.

`EVAL_SELECTED_K` is fixed on dev before final test. Rules/skills and speculative comparisons always use the fixed dev split. The primary test cohort contains all 50 official images, including healthy cases. Zero-shot retains its own prompt and is never substituted for the unified no-tools baseline.

In [ ]:
from dental_agent.harness.notebook_eval import NotebookEvaluations
import json
# Profiles are visible here. Aliases: rtx4090 -> rtx4090_qlora; rtxpro6000_48gb -> workstation48_lora.
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    }
}
# Existing result repository; checkpoints still come from HF_ARTIFACT_REPO.
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"
EVAL_RESULTS_REPO_TYPE = None  # Detect the existing model or dataset repository; never create one.
EVAL_TRACK = "all"  # with_tools, no_tools, or all
EVAL_GRPO_HARNESS_MODES = ["off", "rules_skills_index"]  # Separate trained GRPO cohorts
SPEC_HARNESS_MODE = "rules_skills_index"
SPEC_TARGET_KIND = "grpo"
SPEC_DRAFT_VARIANT = "original"  # original or finetuned; exact merged targets belong in SPEC_CHECKPOINTS
print(json.dumps(EVAL_PROFILES, indent=2))
EVAL_TARGET_PROFILES = "all"  # any key above, a list of keys, or "all"

EVAL_KINDS = ["zeroshot", "baseline", "sft", "grpo", "rules_skills", "speculative"]
EVAL_STAGES = "all"
EVAL_SELECTED_K = 4
EVAL_SPLIT = "test"
EVAL_SIZES = ["canonical", "original"]
DATA_DIR = "data"
DEV_SPLIT = "data/splits/dentex_dev.json"
MODEL_ID = None
MAX_NEW_TOKENS = 16384
MAX_MODEL_LEN = 32768  # total serving capacity; generation remains 16384
SPEC_CHECKPOINTS = {}  # {profile: {exact source adapter prefix: audited merged target}}
VLLM_EXECUTABLE = "vllm"  # isolated serving environment executable
SERVING_GPU_INDICES = "0"
SERVING_PORT = 8910
RUN_MISSING_EVALUATIONS = True  # False: inventory/restore only, no inference

## 4. List available checkpoints and result bundles

In [ ]:
final_evaluations = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=EVAL_KINDS,
    stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K, data_dir=DATA_DIR, dev_split=DEV_SPLIT,
    model_id=MODEL_ID, split=EVAL_SPLIT, sizes=EVAL_SIZES, spec_checkpoints=SPEC_CHECKPOINTS,
    vllm_executable=VLLM_EXECUTABLE, gpu_indices=SERVING_GPU_INDICES, port=SERVING_PORT, serving_context_length=MAX_MODEL_LEN)
AVAILABLE_COHORTS = final_evaluations.list()

## 5. Restore completed evaluations and run missing work
Order: zero-shot → baseline without/with tools → SFT controls → GRPO controls → rules/skills → speculative decoding. Each source checkpoint and profile has its own content identity. Saved episodes resume; complete cohorts skip inference. Failed equivalence gates stay visible and suppress speedup claims. Missing checkpoints or audited merged targets are reported explicitly.

Progress is saved every 60 seconds and after handled interruption/completion. A hard runtime shutdown may lose the unsaved interval. No result bundle contains base weights, optimizer state, credentials or radiograph files.

In [ ]:
FINAL_EVAL_STATUS = final_evaluations.run(execute=RUN_MISSING_EVALUATIONS)
print(json.dumps(FINAL_EVAL_STATUS, indent=2))

## 6. Inspect measured metrics and evidence locations

In [ ]:
from dental_agent.harness.notebook_eval import summarize_inventory
FINAL_SUMMARY = summarize_inventory(FINAL_EVAL_STATUS)
print(json.dumps(FINAL_SUMMARY, indent=2))
print("Local inventory:", final_evaluations.output/"inventory.json")
print("Hub:", "https://huggingface.co/"+("datasets/" if final_evaluations.results_repo_type == "dataset" else "")+final_evaluations.results_repo+"/tree/main/evaluations")

## 7. Optional dev-only K sweep and tool/grounding probes
These preserve the development experiments without changing the final test selection. Each probe has its own identity and Hub bundle.


In [ ]:
RUN_DEV_K_SWEEP = False
RUN_DEV_PROBES = False
DEV_ABLATIONS = [dict(remove_tools=["nudge_crop"], locate_shift_frac=0.0, locate_scale=1.0, seed=42),
                 dict(remove_tools=[], locate_shift_frac=0.2, locate_scale=1.0, seed=42)]
if RUN_DEV_K_SWEEP:
    dev_sweep = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=["sft", "grpo"],
        stages=EVAL_STAGES, selected_k="all", data_dir=DATA_DIR, dev_split=DEV_SPLIT,
        model_id=MODEL_ID, split="dev", sizes=EVAL_SIZES)
    dev_sweep.list()
    DEV_K_STATUS = dev_sweep.run()
if RUN_DEV_PROBES:
    DEV_PROBE_STATUS = []
    for ablation in DEV_ABLATIONS:
        probe = NotebookEvaluations(grpo_harness_modes=EVAL_GRPO_HARNESS_MODES, speculative_harness_mode=SPEC_HARNESS_MODE, speculative_target_kind=SPEC_TARGET_KIND, draft_variant=SPEC_DRAFT_VARIANT, track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES, kinds=["baseline", "sft", "grpo"],
            stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K, data_dir=DATA_DIR, dev_split=DEV_SPLIT,
            model_id=MODEL_ID, split="dev", ablation=ablation)
        DEV_PROBE_STATUS.extend(probe.run())


## 8. Save paired statistics beside the evaluations
This writes JSON and Markdown to the existing artifact repository. No paper assets are generated.


In [ ]:
from dental_agent.harness.notebook_eval import publish_paired_summaries
PAIRED_SUMMARIES = publish_paired_summaries(FINAL_EVAL_STATUS)
print(json.dumps(PAIRED_SUMMARIES, indent=2))
